In [1]:
import tensorflow as tf

In [2]:
!pip install -q -U keras-tuner
import kerastuner as kt


     |████████████████████████████████| 61kB 4.1MB/s 


In [4]:
(img_train, label_train), (img_test, label_test) = tf.keras.datasets.fashion_mnist.load_data()

4423680/4422102 [==============================] - 0s 0us/step


In [5]:
img_train = img_train.astype('float32') / 255.0
img_test = img_test.astype('float32') / 255.0

In [6]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten

#Defining the model

In [8]:
from tensorflow.keras.optimizers import Adam

In [9]:
from tensorflow.keras.losses import SparseCategoricalCrossentropy

In [13]:
def model_builder(hp):
  model = Sequential()
  model.add(Flatten(input_shape=(28,28)))

  #Tune the number of units in the first dense layer
  #Choose an optimal value between 32-512
  hp_units = hp.Int('units', min_value = 32, max_value=512, step=32)
  model.add(Dense(units= hp_units, activation='relu'))
  model.add(Dense(10))

  #Tuning  the learning rate for the optimizer
  # choose an optimal value from 0.01, 0.001, 0.0001
  hp_learning_rate = hp.Choice('learning_rate', values= [1e-2, 1e-3, 1e-4])

  model.compile(optimizer = Adam(learning_rate = hp_learning_rate),
                loss = SparseCategoricalCrossentropy(from_logits=True),
                metrics = ['accuracy'])
  return model

In [14]:
tuner = kt.Hyperband(model_builder,
                     objective = 'val_accuracy',
                     max_epochs = 10,
                     factor = 3,
                     directory = 'my_dir',
                     project_name = 'intro_to_kt'
                     )

#Defining Call backs

In [17]:
import IPython

In [18]:
class ClearTrainingOutput(tf.keras.callbacks.Callback):
  def on_train_end(*args, **kwargs):
    IPython.display.clear_output(wait = True)


In [19]:
tuner.search(img_train, label_train, epochs = 10, validation_data = (img_test, label_test), 
             callbacks = [ClearTrainingOutput()])
best_hps = tuner.get_best_hyperparameters(num_trials = 1)[0]

print(f"""
The hyperparameter search is complete. The optimal number of units in the first densely-connected
layer is {best_hps.get('units')} and the optimal learning rate for the optimizer
is {best_hps.get('learning_rate')}.
""")


INFO:tensorflow:Oracle triggered exit

The hyperparameter search is complete. The optimal number of units in the first densely-connected
layer is 352 and the optimal learning rate for the optimizer
is 0.001.



In [20]:
# Build the model with the optimal hyperparameters and train it on the data
model = tuner.hypermodel.build(best_hps)
model.fit(img_train, label_train, epochs = 10, validation_data = (img_test, label_test))


Epoch 1/10
1875/1875 [==============================] - 7s 4ms/step - loss: 0.4767 - accuracy: 0.8305 - val_loss: 0.4121 - val_accuracy: 0.8530
Epoch 2/10
1875/1875 [==============================] - 7s 4ms/step - loss: 0.3602 - accuracy: 0.8686 - val_loss: 0.3808 - val_accuracy: 0.8616
Epoch 3/10
1875/1875 [==============================] - 7s 4ms/step - loss: 0.3240 - accuracy: 0.8805 - val_loss: 0.3676 - val_accuracy: 0.8646
Epoch 4/10
1875/1875 [==============================] - 7s 4ms/step - loss: 0.3009 - accuracy: 0.8889 - val_loss: 0.3426 - val_accuracy: 0.8734
Epoch 5/10
1875/1875 [==============================] - 7s 4ms/step - loss: 0.2823 - accuracy: 0.8960 - val_loss: 0.3419 - val_accuracy: 0.8782
Epoch 6/10
1875/1875 [==============================] - 7s 4ms/step - loss: 0.2689 - accuracy: 0.9002 - val_loss: 0.3795 - val_accuracy: 0.8651
Epoch 7/10
1875/1875 [==============================] - 7s 4ms/step - loss: 0.2538 - accuracy: 0.9065 - val_loss: 0.3344 - val_accuracy:

In [21]:
model.evaluate(img_test, label_test)

313/313 [==============================] - 1s 2ms/step - loss: 0.3374 - accuracy: 0.8838


[0.3374439775943756, 0.8838000297546387]